In [ ]:
 pip install transformer_lens

In [2]:

import numpy as np
import torch
from transformer_lens import HookedTransformer, patching

# ---------------------------------------------------------------------------
# 0. Setup
# ---------------------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
model.eval()

N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads

# ---------------------------------------------------------------------------
# 1. Dataset: single-digit addition prompt pairs
#
# Restrict operands so the true sum is a single digit (<=9) -- GPT-2's BPE
# tokenizer represents " 0".." 9" as single tokens with a leading space in
# most contexts, which keeps the "correct answer token" well-defined. THIS
# ASSUMPTION NEEDS CHECKING ON FIRST RUN (see sanity check below) -- if it's
# wrong for some operand values, drop those pairs rather than silently
# treating a multi-token answer as single-token.
# ---------------------------------------------------------------------------

def make_pairs(n_pairs=40, seed=0):
    """Balanced, not purely random: enumerate ALL valid (a, b, b_corrupt)
    triples for a,b in [1,4] (48 combos total, since b_corrupt != b),
    shuffle once for a reproducible order, then take the first n_pairs.
    This avoids the earlier version's failure mode where pure random
    sampling could over-represent some operand pairs and skip others
    entirely, making the averaged patching effect partly an artifact of
    which pairs happened to get drawn. If n_pairs >= 48, all combos are
    used exactly once (the true balanced set); if n_pairs < 48, you get a
    reproducible balanced subset, not a resampled one."""
    rng = np.random.default_rng(seed)
    all_combos = [
        (a, b, b_corrupt)
        for a in range(1, 5)
        for b in range(1, 5)
        for b_corrupt in range(1, 5)
        if b_corrupt != b
    ]
    rng.shuffle(all_combos)
    chosen = all_combos[:min(n_pairs, len(all_combos))]
    if n_pairs > len(all_combos):
        print(f"[make_pairs] requested {n_pairs} but only {len(all_combos)} "
              f"unique (a,b,b_corrupt) combos exist for operands in [1,4] -- "
              f"using all {len(all_combos)}, not resampling with repeats.")

    pairs = []
    for a, b, b_corrupt in chosen:
        clean_prompt = f"{a} + {b} ="
        corrupt_prompt = f"{a} + {b_corrupt} ="
        clean_answer = f" {a + b}"
        corrupt_answer = f" {a + b_corrupt}"
        pairs.append((clean_prompt, corrupt_prompt, clean_answer, corrupt_answer))
    return pairs


def sanity_check_single_token_answers(pairs):
    """Drop any pair whose answer isn't a single token for this tokenizer.
    Print how many were dropped -- if it's a large fraction, the whole
    single-digit-restriction assumption above needs rethinking, not just
    silently filtering."""
    kept = []
    dropped = 0
    for clean_p, corrupt_p, clean_a, corrupt_a in pairs:
        ca_ids = model.to_tokens(clean_a, prepend_bos=False)
        xa_ids = model.to_tokens(corrupt_a, prepend_bos=False)
        if ca_ids.shape[-1] == 1 and xa_ids.shape[-1] == 1:
            kept.append((clean_p, corrupt_p, clean_a, corrupt_a))
        else:
            dropped += 1
    print(f"[sanity check] kept {len(kept)}, dropped {dropped} "
          f"(non-single-token answers)")
    return kept


pairs = sanity_check_single_token_answers(make_pairs(n_pairs=60))
assert len(pairs) >= 20, (
    "Too few usable single-token-answer pairs -- the single-digit-answer "
    "assumption may be wrong for this tokenizer/model version. Inspect "
    "model.to_tokens on a few examples before going further."
)

# ---------------------------------------------------------------------------
# 2. Logit-diff metric (same shape as Phase 0's IOI metric: correct-answer
#    logit minus corrupted-answer logit, normalized so 0 = corrupted-run
#    baseline, 1 = clean-run baseline)
# ---------------------------------------------------------------------------

def get_answer_token_ids(clean_answer, corrupt_answer):
    clean_id = model.to_tokens(clean_answer, prepend_bos=False)[0, 0].item()
    corrupt_id = model.to_tokens(corrupt_answer, prepend_bos=False)[0, 0].item()
    return clean_id, corrupt_id


def logit_diff(logits, clean_id, corrupt_id, pos=-1):
    # logits: [batch, seq, vocab]
    return logits[0, pos, clean_id] - logits[0, pos, corrupt_id]


# ---------------------------------------------------------------------------
# 3. Run patching, averaged over all pairs
#
# Two levels, matching Phase 0's approach: coarse (resid_pre, layer x
# position) to find WHERE in the sequence/depth the effect lives, then a
# SEPARATE fine pass (attn head out, layer x head, ALL positions
# aggregated -- not restricted to whatever position the coarse pass
# flagged; see module docstring note on this).
# ---------------------------------------------------------------------------

resid_effects = []
head_effects = []

for clean_prompt, corrupt_prompt, clean_answer, corrupt_answer in pairs:
    clean_id, corrupt_id = get_answer_token_ids(clean_answer, corrupt_answer)

    clean_tokens = model.to_tokens(clean_prompt)
    corrupt_tokens = model.to_tokens(corrupt_prompt)

    # Same-length assumption: "{a} + {b} =" tokenizes identically in length
    # for a,b in [1,4] (single-digit operands). CHECK THIS on first run --
    # if clean/corrupt token counts differ for any pair, patching position
    # alignment breaks silently. Assert defensively:
    if clean_tokens.shape[-1] != corrupt_tokens.shape[-1]:
        continue  # skip misaligned pairs rather than silently mispatch

    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    corrupt_logits = model(corrupt_tokens)

    clean_ld = logit_diff(clean_logits, clean_id, corrupt_id)
    corrupt_ld = logit_diff(corrupt_logits, clean_id, corrupt_id)
    denom = (clean_ld - corrupt_ld)
    if denom.abs() < 1e-4:
        continue  # degenerate pair, would blow up normalization

    def metric(logits, clean_ld=clean_ld, corrupt_ld=corrupt_ld, denom=denom,
               clean_id=clean_id, corrupt_id=corrupt_id):
        ld = logit_diff(logits, clean_id, corrupt_id)
        # NOTE: return the tensor, not a float -- transformer_lens's
        # generic_activation_patch calls .item() on this return value
        # itself, so calling .item() here first would break that (this
        # is exactly the bug hit on first Colab run: 'float' object has
        # no attribute 'item').
        return (ld - corrupt_ld) / denom

    resid_result = patching.get_act_patch_resid_pre(
        model, corrupt_tokens, clean_cache, metric
    )  # [n_layers, seq_len]
    resid_effects.append(resid_result.cpu().numpy())

    head_result = patching.get_act_patch_attn_head_out_all_pos(
        model, corrupt_tokens, clean_cache, metric
    )  # [n_layers, n_heads]
    head_effects.append(head_result.cpu().numpy())

print(f"[patching] usable pairs after alignment/degeneracy filtering: "
      f"{len(resid_effects)} / {len(pairs)}")

resid_effects = np.stack(resid_effects)  # [n_pairs, n_layers, seq_len]
head_effects = np.stack(head_effects)    # [n_pairs, n_layers, n_heads]

resid_mean = resid_effects.mean(axis=0)  # [n_layers, seq_len]
head_mean = head_effects.mean(axis=0)    # [n_layers, n_heads]

best_layer_pos = np.unravel_index(np.argmax(resid_mean), resid_mean.shape)
best_head = np.unravel_index(np.argmax(head_mean), head_mean.shape)

print(f"[patching] strongest resid_pre effect at layer {best_layer_pos[0]}, "
      f"position {best_layer_pos[1]}: {resid_mean[best_layer_pos]:.3f}")
print(f"[patching] strongest single-head effect at layer {best_head[0]}, "
      f"head {best_head[1]}: {head_mean[best_head]:.3f}")

# ---------------------------------------------------------------------------
# 4. Negative control: patch an early, position-irrelevant activation
#    (layer 0 resid_pre at the BOS token) -- should show ~no effect.
# ---------------------------------------------------------------------------

bos_layer0_effect = resid_mean[0, 0]
print(f"[negative control] layer 0 / BOS-token effect: "
      f"{bos_layer0_effect:.3f} (expect ~0)")

# ---------------------------------------------------------------------------
# 5. Independent validation: linear probing at the "=" token, per layer,
#    predicting the sum. Uses a FRESH set of prompts (not the patching set)
#    to avoid any leakage between "where causal patching says the answer
#    lives" and "where a probe finds it."
# ---------------------------------------------------------------------------

def build_probe_dataset(n_examples=200, seed=1):
    rng = np.random.default_rng(seed)
    prompts, sums = [], []
    for _ in range(n_examples):
        a, b = rng.integers(1, 5, size=2)
        prompts.append(f"{a} + {b} =")
        sums.append(a + b)
    return prompts, np.array(sums)


probe_prompts, probe_sums = build_probe_dataset()

# collect residual stream (resid_post) at the final ("=") token, all layers
layer_acts = {layer: [] for layer in range(N_LAYERS)}
for prompt in probe_prompts:
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens)
    for layer in range(N_LAYERS):
        act = cache["resid_post", layer][0, -1, :].cpu().numpy()
        layer_acts[layer].append(act)

from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score

# ridge, not OLS: n_features (768 for gpt2-small) >> n_examples (200),
# OLS would trivially overfit here -- this project has been burned by
# skipping this check before (see decisions.md, overfitting checks on
# Predictive Self). Use CV, not train-set R^2.
#
# alpha is swept, not fixed: a single untested lambda risks the same
# mistake flagged in the Predictive Self review -- one arbitrary
# regularization strength can itself determine the result. Report the
# BEST CV R^2 per layer across the sweep, and separately record which
# alpha won, so a layer that only "works" under one specific alpha is
# visible as a weaker finding than one that's robust across the sweep.
ALPHA_SWEEP = [1.0, 10.0, 100.0]

probe_r2_per_layer = []
probe_best_alpha_per_layer = []
for layer in range(N_LAYERS):
    X = np.stack(layer_acts[layer])
    y = probe_sums
    kfold = KFold(5, shuffle=True, random_state=0)
    alpha_scores = []
    for alpha in ALPHA_SWEEP:
        scores = cross_val_score(Ridge(alpha=alpha), X, y, cv=kfold, scoring="r2")
        alpha_scores.append(scores.mean())
    best_idx = int(np.argmax(alpha_scores))
    probe_r2_per_layer.append(alpha_scores[best_idx])
    probe_best_alpha_per_layer.append(ALPHA_SWEEP[best_idx])
    print(f"[probing] layer {layer}: best CV R^2 = {alpha_scores[best_idx]:.3f} "
          f"(alpha={ALPHA_SWEEP[best_idx]}, all alphas: "
          f"{[round(s, 3) for s in alpha_scores]})")

probe_r2_per_layer = np.array(probe_r2_per_layer)
probe_best_alpha_per_layer = np.array(probe_best_alpha_per_layer)
if len(set(probe_best_alpha_per_layer.tolist())) > 1:
    print("[probing] NOTE: different layers preferred different alphas -- "
          "report per-layer alpha alongside R^2 in the writeup, don't "
          "imply a single regularization strength was used throughout.")
best_probe_layer = int(np.argmax(probe_r2_per_layer))
print(f"[probing] best layer: {best_probe_layer} "
      f"(R^2={probe_r2_per_layer[best_probe_layer]:.3f})")

# ---------------------------------------------------------------------------
# 6. Convergence check (this IS the Phase 1 finding, write it up either way)
# ---------------------------------------------------------------------------

# Compare at the SAME position the probe uses (final/"=" token), not the
# per-layer max over all positions -- .max(axis=1) would let a strong
# effect at some other token (e.g. an operand position) inflate the
# apparent patching effect and make it look falsely convergent/divergent
# with the probe, which only ever sees the "=" token. Also report the
# true per-layer max separately, for the coarse-localization question
# ("where in the sequence does info live at all"), kept distinct from the
# convergence comparison itself.
resid_effect_at_eq = resid_mean[:, -1]        # position-matched to probe
resid_effect_per_layer = resid_mean.max(axis=1)  # best position, any token

print("\n--- Convergence summary (position-matched to probe: '=' token) ---")
print("layer | patching_effect@'=' | probe_R2 | (patching_effect_anywhere)")
for layer in range(N_LAYERS):
    print(f"{layer:5d} | {resid_effect_at_eq[layer]:19.3f} | "
          f"{probe_r2_per_layer[layer]:.3f} | "
          f"({resid_effect_per_layer[layer]:.3f})")

corr = np.corrcoef(resid_effect_at_eq, probe_r2_per_layer)[0, 1]
print(f"\nCorrelation between per-layer patching effect AT '=' and probe "
      f"R^2 (position-matched, the fair comparison): {corr:.3f}")
corr_anywhere = np.corrcoef(resid_effect_per_layer, probe_r2_per_layer)[0, 1]
print(f"(for reference only, NOT the convergence claim: correlation using "
      f"max-over-any-position patching effect instead: {corr_anywhere:.3f})")
print("Do NOT treat a high correlation alone as proof of a discovered "
      "circuit -- write the full per-layer/per-head table, the negative "
      "control value, this correlation, AND the resid_pre/resid_post "
      "~1-layer offset caveat into docs/llm_phase1_addition_circuit.md, "
      "matching decisions.md's existing standard of reporting exact "
      "numbers, not just a verdict.")

# ---------------------------------------------------------------------------
# 7. Save raw results for inclusion in the writeup
# ---------------------------------------------------------------------------

np.savez(
    "phase1_addition_circuit_results.npz",
    resid_mean=resid_mean,
    head_mean=head_mean,
    resid_effect_at_eq=resid_effect_at_eq,
    probe_r2_per_layer=probe_r2_per_layer,
    probe_best_alpha_per_layer=probe_best_alpha_per_layer,
    bos_layer0_control=bos_layer0_effect,
    n_patching_pairs=len(resid_effects),
)
print("\nSaved phase1_addition_circuit_results.npz")


/tmp/ipykernel_789/1253544402.py:12: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
"""
Phase 1 -- self-model-agent LLM track
FOLLOW-UP to phase1_addition_circuit.py, fixing the probing-arm bug found
in the first run: R^2=1.000 at EVERY layer (including layer 0) meant the
probe was memorizing WHICH of only 16 distinct (a,b) combos it saw
(operands restricted to [1,4] for the patching arm's single-token-answer
requirement), not decoding a computed sum. The patching arm's own result
(clean negative control, effect concentrated at layers 8-11 on the "="
position) is NOT affected by this bug and does not need to be re-run.

FIX: the probing arm has no single-token-answer constraint (it never
computes a logit diff against a specific answer token; it only reads
residual-stream activations at "=" and regresses them onto the true sum).
So it can and should use a much larger, independent operand range with
hundreds of DISTINCT (a,b) pairs, making pure memorization implausible as
an explanation for a high R^2. If R^2 is still high on well-separated
train/test combos after this fix, THAT is evidence for a genuinely
decodable representation; if it drops back toward the negative-control
floor for early layers and rises only at later layers (mirroring the
patching curve), that is the convergence this project is actually
looking for.

Run this in the SAME Colab session as phase1_addition_circuit.py (reuses
the loaded `model`), or re-run after loading the model fresh -- either
way, load the saved patching results so they don't need to be recomputed.
"""

import numpy as np
import torch
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score

# ---------------------------------------------------------------------------
# 0. Reuse the already-loaded model if this is the same session; otherwise
#    load fresh. Reuse the saved patching results either way -- no need to
#    redo activation patching, only the probing arm was broken.
# ---------------------------------------------------------------------------

try:
    model  # already loaded in this session?
    print("[setup] reusing already-loaded `model` from this session")
except NameError:
    from transformer_lens import HookedTransformer
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.set_grad_enabled(False)
    model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
    model.eval()
    print("[setup] loaded model fresh")

N_LAYERS = model.cfg.n_layers

saved = np.load("phase1_addition_circuit_results.npz")
resid_mean = saved["resid_mean"]  # [n_layers, seq_len], from the FIRST (valid) run
patching_effect_at_eq = resid_mean[:, -1]  # position-matched to "=" token, as established last time
bos_layer0_control = float(saved["bos_layer0_control"])
print(f"[reuse] loaded patching results from phase1_addition_circuit_results.npz "
      f"(negative control = {bos_layer0_control:.3f})")

# ---------------------------------------------------------------------------
# 1. FIXED probe dataset: large, independent operand range.
#
# No single-token-answer restriction needed here (probing never looks up
# an answer-token id, it regresses onto the raw integer sum) -- so operands
# can range far beyond [1,4]. Using [0,50] each gives 51*51=2601 possible
# ordered pairs; explicitly dedupe to GUARANTEE distinct combos (rather
# than trusting random sampling not to collide), and keep the count well
# below 2601 so there's no risk of exhausting the space and silently
# looping (make_pairs in the original script already showed this failure
# mode once, with the smaller [1,4] range -- same defensive pattern here).
# ---------------------------------------------------------------------------

def build_probe_dataset_large(n_examples=400, seed=1, operand_max=50):
    rng = np.random.default_rng(seed)
    seen = set()
    prompts, sums = [], []
    max_possible = (operand_max + 1) ** 2
    assert n_examples <= max_possible * 0.5, (
        f"requested {n_examples} but operand range only offers {max_possible} "
        f"pairs -- keep well under that to avoid near-exhaustive sampling "
        f"(same silent-degeneracy risk make_pairs already hit once)."
    )
    while len(prompts) < n_examples:
        a, b = rng.integers(0, operand_max + 1, size=2)
        key = (int(a), int(b))
        if key in seen:
            continue
        seen.add(key)
        prompts.append(f"{a} + {b} =")
        sums.append(a + b)
    return prompts, np.array(sums)


probe_prompts, probe_sums = build_probe_dataset_large(n_examples=400, operand_max=50)
print(f"[probe dataset] {len(probe_prompts)} examples, "
      f"{len(set(probe_prompts))} distinct prompts (must equal count above), "
      f"sum range [{probe_sums.min()}, {probe_sums.max()}]")
assert len(set(probe_prompts)) == len(probe_prompts), "duplicate prompts slipped through"

# ---------------------------------------------------------------------------
# 2. Collect activations at the "=" token (same extraction point as before:
#    resid_post, last position) -- only the DATASET changed, not where or
#    how activations are read, so this stays comparable to the original
#    design's intent.
#
# NOTE ON PROMPT LENGTH: unlike the patching arm, the probing arm never
# aligns clean/corrupt pairs, so variable prompt LENGTH across examples
# (e.g. "3 + 4 =" vs "23 + 41 =", different token counts) is fine --
# always indexing position -1 (the actual last token, "=") is length-
# agnostic and correct regardless.
# ---------------------------------------------------------------------------

layer_acts = {layer: [] for layer in range(N_LAYERS)}
for prompt in probe_prompts:
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens)
    for layer in range(N_LAYERS):
        act = cache["resid_post", layer][0, -1, :].cpu().numpy()
        layer_acts[layer].append(act)

# ---------------------------------------------------------------------------
# 3. Same ridge+CV+alpha-sweep protocol as the (already-fixed) first run.
# ---------------------------------------------------------------------------

ALPHAS = [1.0, 10.0, 100.0]
probe_r2_per_layer = []
for layer in range(N_LAYERS):
    X = np.stack(layer_acts[layer])
    y = probe_sums
    best_alpha_score = -np.inf
    all_scores = []
    for alpha in ALPHAS:
        scores = cross_val_score(Ridge(alpha=alpha), X, y,
                                  cv=KFold(5, shuffle=True, random_state=0), scoring="r2")
        all_scores.append(scores.mean())
        best_alpha_score = max(best_alpha_score, scores.mean())
    probe_r2_per_layer.append(best_alpha_score)
    print(f"[probing, FIXED dataset] layer {layer}: best CV R^2 = {best_alpha_score:.3f} "
          f"(all alphas: {all_scores})")

probe_r2_per_layer = np.array(probe_r2_per_layer)
best_probe_layer = int(np.argmax(probe_r2_per_layer))
print(f"[probing, FIXED dataset] best layer: {best_probe_layer} "
      f"(R^2={probe_r2_per_layer[best_probe_layer]:.3f})")

if probe_r2_per_layer[0] > 0.5:
    print("\n[WARNING] layer 0 R^2 is still high -- this would be surprising for a "
          "genuinely computed sum (layer 0 is just the embedding + one block) and "
          "suggests the memorization concern may not be fully resolved even with "
          "400 distinct pairs. Inspect further (e.g. check whether the model's "
          "embedding alone linearly encodes small integers well, which is a known, "
          "separate phenomenon from computing a+b) before writing up a convergence "
          "claim.")

# ---------------------------------------------------------------------------
# 4. Re-run the convergence comparison with the FIXED probe curve against
#    the UNCHANGED, already-valid patching curve.
# ---------------------------------------------------------------------------

print("\n--- Convergence summary (FIXED probing dataset) ---")
print("layer | patching_effect@'=' | probe_R2 (large operand range)")
for layer in range(N_LAYERS):
    print(f"{layer:5d} | {patching_effect_at_eq[layer]:20.3f} | {probe_r2_per_layer[layer]:.3f}")

corr = np.corrcoef(patching_effect_at_eq, probe_r2_per_layer)[0, 1]
print(f"\nCorrelation between per-layer patching effect @ '=' and FIXED probe R^2: {corr:.3f}")
print("This number (not the one from the first run) is the one to trust and write "
      "into docs/llm_phase1_addition_circuit.md, together with the layer-0 sanity "
      "check above and an explicit note of what changed (operand range 1-4 -> 0-50, "
      "16 possible combos -> 400 distinct sampled combos out of 2601 possible).")

np.savez(
    "phase1_addition_circuit_results_probe_fixed.npz",
    resid_mean=resid_mean,
    probe_r2_per_layer=probe_r2_per_layer,
    patching_effect_at_eq=patching_effect_at_eq,
    bos_layer0_control=bos_layer0_control,
    correlation_fixed=corr,
)
print("\nSaved phase1_addition_circuit_results_probe_fixed.npz")